# Writing GRIB to geopoints

In [1]:
# get input GRIB data
import earthkit.data as ekd

ds = ekd.from_source("sample", "test.grib").to_fieldlist()

test.grib:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

#### Using to_target() on the data object

In [2]:
# writing a field
ds[0].to_target("file", "_res_t.gpt")

# writing a whole fieldlist
ds.to_target("file", "_res_t.gpt")

The output is a plain text file. The rows below the `#DATA` line are the points of the field.

In [3]:
# the beginning of the file the fieldlist produced
with open("_res_t.gpt") as f:
    print("".join(f.readlines()[:8]), end="")

#GEOPOINTSET
#GEO
#METADATA
param=2t
level_type=surface
#DATA
73 -27 0 20200513 1200 262.7802734375
73 -23 0 20200513 1200 267.447265625


#### Reading the file back

In [4]:
ds.to_target("file", "_res_t.gpt")

back = ekd.from_source("file", "_res_t.gpt").to_fieldlist()
back.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,2t,2020-05-13 12:00:00,2020-05-13 12:00:00,0 days,0,surface,None,_unstructured
1,msl,2020-05-13 12:00:00,2020-05-13 12:00:00,0 days,0,surface,None,_unstructured


The fields we read back are the ones we wrote: same number of fields, same param, same points and same values.

#### Using a Target object

In [5]:
# basic usage
target = ekd.create_target("file", "_res_t.gpt")
target.write(ds)
target.close()

# can be used as a context manager, no need to call close() in the end
with ekd.create_target("file", "_res_t.gpt") as target:
    target.write(ds)

# a fieldlist can be written field by field into the target
with ekd.create_target("file", "_res_t.gpt") as target:
    for f in ds:
        target.write(f)

#### Using encoders

In [6]:
# the suffix of the target selects the encoder
ds.to_target("file", "_res_t.gpt")

# explicitly specifying the encoder
ds.to_target("file", "_res_t.gpt", encoder="geopoints")

# using an encoder object
encoder = ekd.create_encoder("geopoints")
ds.to_target("file", "_res_t.gpt", encoder=encoder)

In [7]:
encoder = ekd.create_encoder("geopoints", metadata={"author": "earthkit"})
with ekd.create_target("file", "_res_t.gpt") as target:
    target.write(ds, encoder=encoder)

with open("_res_t.gpt") as f:
    print("".join(f.readlines()[:7]), end="")

#GEOPOINTSET
#GEO
#METADATA
author=earthkit
param=2t
level_type=surface
#DATA


#### Pressure level fields

In [8]:
pl = ekd.from_source("sample", "tuv_pl.grib").to_fieldlist()
pl.to_target("file", "_res_pl.gpt")

# the beginning of the file
with open("_res_pl.gpt") as f:
    print("".join(f.readlines()[:8]), end="")

tuv_pl.grib:   0%|          | 0.00/4.22k [00:00<?, ?B/s]

#GEOPOINTSET
#GEO
#METADATA
param=t
level_type=pressure
#DATA
90 0 1000 20180801 1200 272.5641784667969
90 30 1000 20180801 1200 272.5641784667969


Read back, every field knows its level type and sits on the same pressure level as the field we wrote.

In [9]:
back = ekd.from_source("file", "_res_pl.gpt").to_fieldlist()

assert len(back) == len(pl)

for src, tgt in zip(pl, back):
    assert tgt.get("parameter.variable") == src.get("parameter.variable")
    assert tgt.get("vertical.level_type") == "pressure"
    assert tgt.get("vertical.level") == src.get("vertical.level")

levels = sorted({f.get("vertical.level") for f in back})
print(f"{len(back)} fields on pressure levels {levels}: level types and levels are identical")

18 fields on pressure levels [300, 400, 500, 700, 850, 1000]: level types and levels are identical
